# DPO
[Source](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/04_preference-tuning-with-dpo/dpo-from-scratch.ipynb)

## 0 · Goal & config
- **Mac:** local `.venv` kernel; run from `Training/`.
- **Runpod:** keep [setup.ipynb](../Runpod/setup.ipynb) on the local kernel and run **Start**. Here, choose **Select Kernel → Select Another Kernel → Existing Jupyter Server**, paste the URL, then choose **Python 3**. The first code cell sets the folder; device should print `cuda`.
- **Runpod checkpoint:** upload the instruction checkpoint with setup's `upload_checkpoint(...)` before selecting it below.

In [ ]:
from pathlib import Path

# Use the copied project folder on Runpod.
pod_folder = Path("/workspace/LLM-fine-tuning/Training")
if pod_folder.is_dir():
    %cd {pod_folder}

#### Imports

In [ ]:
from functools import partial

import tiktoken
import torch
from torch.utils.data import DataLoader, Dataset

from reference.configs import gpt2_BASE_CONFIG, gpt2_model_configs
from reference.data import load_dpo_data
from reference.generate import generate
from reference.model_io import text_to_token_ids, token_ids_to_text
from reference.models.gpt2 import GPT2Model


#### Model config

In [ ]:
# Options: gpt2-small (124M), gpt2-medium (355M), gpt2-large (774M), gpt2-xl (1558M).
CHOOSE_MODEL = "gpt2-medium (355M)"

# ** unpacks dictionary entries; later values win on duplicate keys.
model_config = {**gpt2_BASE_CONFIG, **gpt2_model_configs[CHOOSE_MODEL]}

#### Run folder

In [ ]:
folder = Path("runs/dpo/001_baseline")
checkpoint_path = max(Path("runs/instruction/002_finetuned").glob("checkpoint_*.pth"))


#### Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")
print("device:", device)

#### Tokenizer

In [105]:
tokenizer = tiktoken.get_encoding("gpt2")
tokenizer.n_vocab, tokenizer.eot_token

(50257, 50256)

## 1 · Load model
Load the instruction-finetuned checkpoint.

In [ ]:
%%capture

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model_config = checkpoint.get("model_config", model_config)

model = GPT2Model(model_config)
model.load_state_dict(checkpoint["model"])
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6, weight_decay=0.01)
model.eval()
del checkpoint


## 2 · Data + DataLoader
Load data, prepare token IDs, and build batches.

In [9]:
data = load_dpo_data()
data[:1]

[{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
  'input': 'freind --> friend',
  'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".',
  'rejected': 'The spelling of the given phrase "freind" is flat out wrong, get it together, the correct spelling is "friend".',
  'chosen': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]

#### Creating training, validation, and test splits    
Next, we divide the dataset into 3 subsets, 85% training data, 5% validation data, and 10% test data:

In [109]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


In [10]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}")

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction_text + input_text

desired_response = f"### Response:\n{data[0]['chosen']}" # 'rejected'
desired_response

'### Response:\nThe spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'

**Dataset groups by example → batch groups by key.**

### Dataset: list of dictionaries

```text
[
  {"prompt": P1, "chosen": C1, "rejected": R1},
  {"prompt": P2, "chosen": C2, "rejected": R2}
]
```

### After collate: one batch dictionary

```text
{
  "prompt":        [tensor(P1), tensor(P2)],
  "chosen":        tensor([C1, C2]),
  "rejected":      tensor([R1, R2]),
  "chosen_mask":   tensor([mask_C1, mask_C2]),
  "rejected_mask": tensor([mask_R1, mask_R2])
}
```

`P/C/R` represent token sequences. Responses are padded before stacking.

**Each tensor row = one example. `prompt` stays a list of tensors.**


In [ ]:
class PreferenceDataset(Dataset):

    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_data = []
        for entry in data:
            prompt = format_input(entry)
            combined_chosen = f"{prompt}\n\n### Response:\n{entry['chosen']}"
            combined_rejected = f"{prompt}\n\n### Response:\n{entry['rejected']}"

            encoded_prompt    = tokenizer.encode(prompt)
            encoded_chosen    = tokenizer.encode(combined_chosen)
            encoded_rejected  = tokenizer.encode(combined_rejected)
            
            self.encoded_data.append({
                "prompt":encoded_prompt,
                "chosen": encoded_chosen,
                "rejected": encoded_rejected
            })
    
    def __getitem__(self, index):
        return self.encoded_data[index]
    
    def __len__(self):
        return len(self.data)



dataset = PreferenceDataset(data, tokenizer)
for key, values in dataset.__getitem__(0).items():
    print(key, len(values))

In [ ]:
def custom_collate_fn(batch, pad_token = 50256, 
                      allowed_max_length= None, 
                      mask_prompt_tokens = True,  device="cpu"):

    # Initialize lists to hold batch data
    batch_data = {
        "prompt": [],
        "chosen": [],
        "rejected": [],
        "rejected_mask": [],
        "chosen_mask": []
    }

    # Determine the longest sequence to set a common padding length
    max_length_common = 0
    if batch:
        for key in ["chosen", "rejected"]:
            current_max = max([len(item[key])+1 for item in batch])
            max_length_common = max(max_length_common, current_max)
        

    # Process each item in the batch: padding, tensor, 
    for item in batch:
        prompt_tensor = torch.tensor(item["prompt"])
        batch_data["prompt"].append(prompt_tensor)
        
        for key in ["chosen", "rejected"]:
            
            # Adjust padding according to the common maximum length
            padded_item = torch.tensor(item[key] + [pad_token] * (max_length_common - len(item[key])))
            batch_data[key].append(padded_item)

            mask = torch.ones(len(padded_item)).bool()
             # Set mask for all padding tokens to False
            mask[len(item[key]):] = False

            # Set mask for all input tokens to False
            # +2 sets the 2 newline ("\n") tokens before "### Response" to False
            if mask_prompt_tokens:
                mask[:len(prompt_tensor)+2] = False

            batch_data[f"{key}_mask"].append(mask)


    for key in ["chosen", "chosen_mask", "rejected", "rejected_mask"]:
        # Stack all sequences into a tensor for the given key
        torch_stack = torch.stack(batch_data[key])

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
                torch_stack = torch_stack[:,:allowed_max_length]

        batch_data[key] = torch_stack.to(device)

    return batch_data


In [ ]:
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,            # Put the data directly on a GPU if available
    mask_prompt_tokens=True,  # This is optional
    allowed_max_length=1024   # The supported context length of the model
)


In [99]:
example_data = data[:2]

example_dataset = PreferenceDataset(example_data, tokenizer)

example_dataloader = DataLoader(
    example_dataset,
    batch_size=2,
    collate_fn=customized_collate_fn,
    shuffle=False
)

for batch in example_dataloader:
    break

print("batch.keys:", batch.keys())
batch["prompt"]

batch.keys: dict_keys(['prompt', 'chosen', 'rejected', 'rejected_mask', 'chosen_mask'])


[tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198,    36,  2100,  4985,   262,  1708,  9546,
           416, 25449,   340,   656,   262, 24993,  1813,    13,   198,   198,
         21017, 23412,    25,   198, 19503,   521, 14610,  1545]),
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
           257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
         21017, 46486,    25,   198, 18378,   262,  1708,  6827,   329, 23491,
            13,   198,   198, 21017, 23412,    25,   198,  1544,   467,   284,
           262,  3952,   790,  1110,    13])]

In [106]:
def decode_tokens_from_batch(token_ids, tokenizer):
    flat_tensor = token_ids.flatten().tolist()
    print(tokenizer.decode(flat_tensor))

decode_tokens_from_batch(batch["chosen"][0], tokenizer)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [108]:
print("chosen inputs:", batch["chosen"][0].shape)
print("chosen mask:  ", batch["chosen_mask"][0].shape)

chosen inputs: torch.Size([81])
chosen mask:   torch.Size([81])


In [104]:
text = decode_tokens_from_batch(
    token_ids=batch["chosen"][0][batch["chosen_mask"][0]],
    tokenizer=tokenizer,
)
print(text)

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".
None


#### Creating training, validation, and test set data loaders

    Since we applied padding on a batch-by-batch basis, each row has a different shape
    This is for efficiency reasons because it would be inefficient to pad all samples to the longest sample in the whole dataset

In [110]:
num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = PreferenceDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)


val_dataset = PreferenceDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = PreferenceDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

print("Train loader:")
for batch in train_loader:
    print(
        batch["chosen"].shape,
        batch["rejected"].shape,
    )
    

Train loader:
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 81]) torch.Size([8, 81])
torch.Size([8, 94]) torch.Size([8, 94])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 99]) torch.Size([8, 99])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 88]) torch.Size([8, 88])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 97]) torch.Size([8, 97])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 89]) torch.Size([8, 89])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 84]) torch.Size([8, 84])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 101]) torch.Size([8, 101])
torch.Size([8, 87]) torch.Size([8, 87])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 80]) torc

## 3 · Loss / objective

## 4 · Generate / predict
Generate a starting response before this training run.

In [ ]:
torch.manual_seed(123)
model.eval()
sample_input = format_input(val_data[0])
token_ids = generate(model, 
idx = text_to_token_ids(sample_input, tokenizer).to(device), 
max_new_tokens = 35,
context_length = model_config["context_length"],
eos_id = tokenizer.eot_token)

generated_text = token_ids_to_text(token_ids, tokenizer)
response_text = generated_text[len(sample_input):].strip()
print(f"{response_text}")